# Datasets auf gemeinsamen Zeitraum trimmen

**Stichtag:** 2001-10-01  
**Ausgeschlossen:** ALC (2019), AMS (2004), PGHN (2006) — zu kurzer Zeitraum  
**Ergebnis:** 26 Aktien + SSMI, alle ab 2001-10-01 bis 2021-05-17  

Die Original-CSVs werden **nicht** verändert. Es werden neue trimmed-CSVs in `DataSets/trimmed/` gespeichert.

In [1]:
import pandas as pd
import os

In [2]:
CUTOFF_DATE = "2001-10-01"
EXCLUDE_TICKERS = {"ALC", "AMS", "PGHN"}
DATA_DIR = "DataSets"
OUTPUT_DIR = os.path.join(DATA_DIR, "trimmed")

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# Alle CSVs laden (ohne ausgeschlossene Ticker)
csv_files = sorted(
    f for f in os.listdir(DATA_DIR)
    if f.endswith(".csv") and f.replace(".csv", "") not in EXCLUDE_TICKERS
)

print(f"{len(csv_files)} Dateien gefunden (ALC, AMS, PGHN ausgeschlossen)")
print(f"Ticker: {[f.replace('.csv','') for f in csv_files]}")

27 Dateien gefunden (ALC, AMS, PGHN ausgeschlossen)
Ticker: ['ABBN', 'ADEN', 'BAER', 'CFR', 'CLN', 'CSGN', 'GEBN', 'GIVN', 'KNIN', 'LOGN', 'LONN', 'NESN', 'NOVN', 'ROG', 'SCHP', 'SCMN', 'SGSN', 'SIKA', 'SLHN', 'SOON', 'SREN', 'SSMI', 'STMN', 'TEMN', 'UBSG', 'UHR', 'ZURN']


In [4]:
# Trimmen und als neue CSVs speichern
summary_rows = []

for filename in csv_files:
    ticker = filename.replace(".csv", "")
    df = pd.read_csv(os.path.join(DATA_DIR, filename), parse_dates=["Date"], index_col="Date")
    
    original_len = len(df)
    original_start = df.index.min()
    
    # Nur Daten ab Stichtag behalten
    df_trimmed = df.loc[CUTOFF_DATE:].copy()
    
    # Als neue CSV speichern
    df_trimmed.to_csv(os.path.join(OUTPUT_DIR, filename))
    
    summary_rows.append({
        "Ticker": ticker,
        "Original Start": original_start.strftime("%Y-%m-%d"),
        "Original Zeilen": original_len,
        "Trimmed Start": df_trimmed.index.min().strftime("%Y-%m-%d"),
        "Trimmed Ende": df_trimmed.index.max().strftime("%Y-%m-%d"),
        "Trimmed Zeilen": len(df_trimmed),
        "Entfernt": original_len - len(df_trimmed),
    })

summary = pd.DataFrame(summary_rows)
summary

,Ticker,Original Start,Original Zeilen,Trimmed Start,Trimmed Ende,Trimmed Zeilen,Entfernt
0,ABBN,1999-06-28,5570,2001-10-01,2021-05-17,4980,590
1,ADEN,1999-05-10,5605,2001-10-01,2021-05-17,4980,625
2,BAER,2000-01-03,5435,2001-10-01,2021-05-17,4980,455
3,CFR,1995-04-10,6669,2001-10-01,2021-05-17,4980,1689
4,CLN,1996-03-29,6416,2001-10-01,2021-05-17,4980,1436
5,CSGN,1995-04-14,6666,2001-10-01,2021-05-17,4980,1686
6,GEBN,1999-03-29,5635,2001-10-01,2021-05-17,4980,655
7,GIVN,2000-01-03,5435,2001-10-01,2021-05-17,4980,455
8,KNIN,2000-05-22,5335,2001-10-01,2021-05-17,4980,355
9,LOGN,2000-01-03,5435,2001-10-01,2021-05-17,4980,455


In [5]:
# Verifizieren: alle trimmed Datasets haben gleichen Zeitraum
starts = summary["Trimmed Start"].unique()
ends = summary["Trimmed Ende"].unique()

print(f"Einheitlicher Start: {starts}")
print(f"Einheitliches Ende:  {ends}")
print(f"Zeilenbereich:       {summary['Trimmed Zeilen'].min()} – {summary['Trimmed Zeilen'].max()}")
print(f"\nTotal {len(summary)} trimmed CSVs gespeichert in '{OUTPUT_DIR}/'.")

Einheitlicher Start: ['2001-10-01']
Einheitliches Ende:  ['2021-05-17']
Zeilenbereich:       4934 – 4980

Total 27 trimmed CSVs gespeichert in 'DataSets/trimmed/'.
